In [ ]:
import os
import json
import gc

import matplotlib.pyplot as plt
import matplotlib

try:
    this_file = os.path.dirname(__file__)
except NameError:
    this_file = os.getcwd()

from eva_experiments.mg_io.data_loader import import_database, DEFAULT_DIRS
from eva_experiments.mg_io.accession_db import AccessionDatabase
from eva_experiments.mg_io.accession import Accession

export_path = '../exports/dataset/v2'

# matplotlib's interactive backend has a memory leak, 
# use this for rendering
plt.ioff()
matplotlib.use('Agg')

# INPUT

In [ ]:
# point BASE_DIR to a directory that contains a directory called 'mg-data'
# All subdirectories under mg-data is used for analysis
BASE_DIR = os.path.join(os.path.dirname(this_file),'Database','mg-data')
ANALYSIS_DICT = {'gain':{'do_clamp':False},
                 'velocity':{'value_smoothing':3},
                 'speed':{'value_smoothing':3},
                 'acceleration':{'value_smoothing':3}}

AccDatabase = AccessionDatabase(import_database(BASE_DIR, DEFAULT_DIRS[:2]))

In [ ]:
figures_path = '../exports/figures'
try:
    os.mkdir(figures_path)
except:
    pass

accessions = list(AccDatabase.accessions.values())
del AccDatabase

while accessions:
    print(f'{len(accessions)} left')

    acc = accessions.pop()
    acc:Accession = acc.analyse(ANALYSIS_DICT).standardise_cycles()

    print('\tanalysis and standardisation finished; beginning save_path generation')

    cycle_dur = int(acc.standard_cycle_length/120)

    pathology_path = acc.pathology_class
    patient_path = acc.accession_str[:5]
    acc_path = acc.accession_str[7:]
    figure_path = figures_path
    json_path = export_path

    for path in (pathology_path,patient_path):
        json_path = os.path.join(json_path, path)
        try:
            os.mkdir(json_path)
        except:
            pass
    
    for path in (pathology_path,patient_path,acc_path):
        figure_path = os.path.join(figure_path, path)
        try:
            os.mkdir(figure_path)
        except:
            pass
    print('\tfigure_path and json_path generated; beginning json saving')

    json_info = {
        'patient_name':acc.patient_name,
        'pathology_class':acc.pathology_class,
        'visit_date':acc.visit_date,
        'axis':acc.axis,
        'frequency':acc.frequency}\
            |{k:getattr(acc,k) for k in 
        ['standard_gain_dict','standard_velocity_dict','standard_speed_dict',
         'standard_acceleration_dict','standard_analysis',
         'standard_energy_dict','standard_power_dict']
    }

    json_file_path = os.path.join(json_path,f'{acc.accession_str}.json')

    # if os.path.isfile(json_file_path):
    with open(json_file_path,'w') as json_file:
        json.dump(json_info, json_file)
    
    del json_info
    print('\tjson saved, beginning figure saving')

    #PER CYCLE FIGURE
    end_time = acc.standard_cycle_duration
    for start in range(0,end_time,cycle_dur):
        end = start + cycle_dur
        print(f'\t\t{start} - {end} / {end_time} begin')

        figure_name = f'{start:02d}-{end:02d} sec'

        if os.path.isfile(os.path.join(figure_path,f'{figure_name}.jpg')):
            print('\t\t\tskipped')
            continue

        acc.draw(attributes_to_draw=['position','gain','speed','velocity','acceleration'],
                 region=(start,cycle_dur),
                 tick_dist=0.25,
                 use_standard=True,
                 save_name=figure_name,
                 save_dir=figure_path,
                 do_draw=False)
        print(f'\t\t\t{start} - {end} / {end_time} ended')
    
    # ENTIRE RECORDING FIGURE
    figure_name = '0000_full'
    full_fig_path = os.path.join(figure_path,f'{figure_name}.jpg')

    if os.path.isfile(full_fig_path):
        os.remove(full_fig_path)
        acc.draw(['position','gain','speed','velocity','acceleration'], tick_dist=10,
                 figsize=(35,15), use_standard=True, save_dir=save_path,save_name=save_name,do_draw=False)
    
    # RAM STUFF
    del acc
    gc.collect()

    print(f'this finished \n')

In [ ]:
# # Old

# normalised_patient_holder = {}
# for acc_idx, acc_data in enumerate(process_n_split.values()):
#     # if len(normalised_patient_holder)>1:
#     #     break
#     acc_data: Accession
#     acc_info =  acc_data.accession

#     print(f'({acc_idx+1}/{len(process_n_split)})  STARTING {acc_info}')
#     if not hasattr(acc_data, 'split'):
#         print(f'└─  NO SPLIT, SKIPPED {acc_info}','+'*30,sep="\n")
#         continue
#     if acc_data.file_idx != 1:
#         print(f'└─  NOT 0.75Hz, SKIPPED {acc_info}','+'*30,sep="\n")
#         continue

#     axis_info = acc_data.axis
#     patient_id = '-'.join(f'{d:02d}' for d in acc_info[:2])
#     freq_info = 'freq_' + str([0.5,0.75,1.0][acc_info[-1]])
#     point_amt = [480,360,240][acc_info[-1]]

#     # top layer
#     default_data = {
#         'patient_name':acc_data.patient_name,
#         'pathology_class':acc_data.pathology_class,
#         'stored':[],
#         'visits':{}
#     }

#     normalised_patient_holder[patient_id] = normalised_patient_holder.setdefault(patient_id,default_data)
#     if acc_info in normalised_patient_holder[patient_id]['stored']:
#         print('└─ SKIPPED, ALREADY IN STORAGE','+'*30,sep="\n")
#         continue

#     top_layer = normalised_patient_holder[patient_id]

#     visit_data = {
#         'visit_date':acc_data.visit_date,
#     }

#     middle_layer:dict

#     # find visit for middle layer
#     for visit_idx, visit in top_layer['visits'].items():
#         if visit['visit_date'] == acc_data.visit_date:
#             middle_layer = top_layer['visits'][visit_idx]
#             break

#     else:
#         visit_idx = len(top_layer['visits'])
#         top_layer['visits'][visit_idx] = visit_data
#         middle_layer = top_layer['visits'][visit_idx]

#     middle_layer[freq_info] = middle_layer.setdefault(freq_info,{})
#     final_layer = middle_layer[freq_info]

#     # store data in final layer
#     wanted_data = {
#         f'velocity_{axis_info}':acc_data.split['velocity'],
#         f'speed_{axis_info}':acc_data.split['speed'],
#         f'gain_{axis_info}':acc_data.split['gain']
#     }
    
#     for k,v in wanted_data.items():
#         complete_recording = {

#         }
#         for section_idx, section in enumerate(v):
#             # for cycle_key, cycle_val in section.items():
#             #     section_name = f'section_{section_idx:02d}'
#             #     dir_name = cycle_key if cycle_key=='whole' else f'{cycle_key}_half'
#             #     use_point_amt = point_amt if cycle_key=='whole' else int(point_amt/2)
#             #     cycle_key = dir_name + '_cycle'
#             #     final_layer[cycle_key] = final_layer.setdefault(cycle_key,{})
#             #     final_layer[cycle_key][section_name] = final_layer[cycle_key].setdefault(section_name, {})
#             #     final_layer[cycle_key][section_name][k] = cycle_val[:use_point_amt]

#             for cycle_key, cycle_val in section.items():
#                 if cycle_key != 'whole':
#                     continue
#                 dir_name = cycle_key if cycle_key=='whole' else f'{cycle_key}_half'
#                 use_point_amt = point_amt if cycle_key=='whole' else int(point_amt/2)
#                 cycle_key = dir_name + '_cycle'
#                 complete_recording[cycle_key] = complete_recording.setdefault(cycle_key,[]) + cycle_val[:use_point_amt]
#         final_layer[k] = complete_recording


#     top_layer['stored'].append(acc_info)
    
#     print(f'└─ FINISHED {acc_info}','+'*30,sep="\n")

# stripped_patient_holder = {}
# for patient_k, patient_v in normalised_patient_holder.items():
#     check_axes = [store[3] for store in patient_v['stored']]
#     check_attrs = [entry for visit in patient_v['visits'].values()
#                    for entry in visit['freq_0.75']]
#     try:
#         assert check_axes.count(0) == check_axes.count(1)
#         assert all(attr in check_attrs for attr in [f'{a}_{d}' for a in ('velocity','speed','gain') for d in ('horizontal','vertical')])
#         stripped_patient_holder[patient_k] = patient_v
#     except:
#         print(f'removed {patient_k} because axes imbalanced')
#         continue